# Score GenD variants on Celeb-DF crops built two ways

Scores the authors' GenD checkpoints (MIT) on the Celeb-DF v2 test videos, cropped two ways:

- the lab's square crop (16 frames per video, 1.5x box, reframed to 1.3x with `tighten_crop`);
- GenD's own framing (32 frames per video, faces aligned on 5 landmarks at 1.3x, `faces.aligned_crop`), from `build_celebdf_test.ipynb` with `ALIGN = True`.

Checkpoints: `GenD_CLIP_L_14` and `GenD_PE_L` (Perception Encoder), and their ensemble, the mean of the two fake probabilities. The paper reports 0.960 (CLIP) and 0.958 (PE) Celeb-DF video AUC. Reports go to `HF_MODEL/pretrained/<detector>/<test>.json`.

**Runtime:** Colab Pro, L4 or A100. **Needs** the Colab secret `HF_TOKEN`.

### Configuration

In [ ]:
REPO_SOURCE = "git+https://github.com/paulohenriquevn/face-decision-lab@workspace"
HF_MODEL = "paulohenriquevn/face-decision-clip-ln"
TEST_SETS = {
    # name -> (dataset, face margin its crops hold)
    "cross_celebdf": ("paulohenriquevn/face-decision-celebdf-test", 1.5),
    "cross_celebdf_aligned": ("paulohenriquevn/face-decision-celebdf-test-aligned", 1.3),
}
GEND_MODELS = {"gend_clip_l14": "yermandy/GenD_CLIP_L_14", "gend_pe_l": "yermandy/GenD_PE_L"}
GEND_FACE_SCALE = 1.3

### Installation

In [ ]:
%%capture
!pip install -q "face-decision-lab @ {REPO_SOURCE}" "timm>=1.0.20"

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token = userdata.get("HF_TOKEN"))

### Models

`GenD.from_pretrained` fails under transformers 5, which builds models on the meta device where GenD's `__init__` cannot load its backbone, so each model is built directly from its own `modeling_gend.py` and `model.safetensors`. The model card's examples (a Deepfakes fake, then a real frame) confirm which logit means fake.

In [ ]:
import importlib.util, io, json
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import requests
import torch
from huggingface_hub import HfApi, hf_hub_download
from PIL import Image
from safetensors.torch import load_file
from tqdm.auto import tqdm
from face_decision_lab.faces import tighten_crop
from face_decision_lab.hub import load_hub_samples
from face_decision_lab.protocol import evaluation_report
from face_decision_lab.rows import open_image

api = HfApi()
pool = ThreadPoolExecutor(max_workers = 8)
EXAMPLES = ["https://github.com/yermandy/deepfake-detection/blob/main/datasets/FF/DF/000_003/000.png?raw=true",
            "https://github.com/yermandy/deepfake-detection/blob/main/datasets/FF/real/000/000.png?raw=true"]
example_frames = [Image.open(io.BytesIO(requests.get(u, timeout = 30).content)).convert("RGB") for u in EXAMPLES]

def load_gend(repo):
    spec = importlib.util.spec_from_file_location(f"modeling_{repo.split('/')[1]}", hf_hub_download(repo, "modeling_gend.py"))
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    model = module.GenD(module.GenDConfig.from_pretrained(repo))
    loaded = model.load_state_dict(load_file(hf_hub_download(repo, "model.safetensors")), strict = False)
    assert not loaded.missing_keys and not loaded.unexpected_keys, loaded
    model = model.cuda().eval()
    with torch.no_grad():
        x = torch.stack([model.feature_extractor.preprocess(f) for f in example_frames]).cuda()
        fake = model(x).softmax(-1)[:, 1].tolist()
    assert fake[0] > 0.5 > fake[1], (repo, fake)
    print(repo, "examples (fake, real):", fake)
    return model

models = {name: load_gend(repo) for name, repo in GEND_MODELS.items()}

### Score

In [ ]:
KEYS = ("samples", "auc", "eer", "bpcer_at_max_apcer", "videos", "video_auc")

def scores_of(model, part, images, crop_scale):
    def prepare(sample):
        return model.feature_extractor.preprocess(tighten_crop(open_image(images[sample.image_path]), GEND_FACE_SCALE, crop_scale))
    out = []
    with torch.no_grad(), torch.autocast("cuda", dtype = torch.bfloat16):
        for i in tqdm(range(0, len(part), 128), leave = False):
            x = torch.stack(list(pool.map(prepare, part[i:i + 128]))).cuda()
            out += model(x).float().softmax(-1)[:, 1].cpu().tolist()
    return out

def report(detector, test, part, scores, **extra):
    result = {k: v for k, v in evaluation_report(part, scores).items() if k in KEYS} | extra
    api.upload_file(path_or_fileobj = json.dumps(result, indent = 2).encode(), repo_id = HF_MODEL,
                    path_in_repo = f"pretrained/{detector}/{test}.json", commit_message = f"{detector} on {test}")
    print(detector, test, json.dumps({k: result[k] for k in ("auc", "eer", "video_auc")}))

for test, (repo, crop_scale) in TEST_SETS.items():
    part, part_images = load_hub_samples(repo, key_prefix = "celebdf")
    images = dict(zip([s.image_path for s in part], part_images, strict = True))
    by_model = {name: scores_of(model, part, images, crop_scale) for name, model in models.items()}
    for name, scores in by_model.items():
        report(name, test, part, scores, face_scale = GEND_FACE_SCALE, crop_scale = crop_scale)
    ensemble = np.mean([by_model[name] for name in models], axis = 0).tolist()
    report("gend_ensemble_clip_pe", test, part, ensemble, members = list(models), crop_scale = crop_scale)